## How to run in Colab
1. Open [Google Colab](https://colab.research.google.com) → **File → Upload notebook** → choose this file.
2. Download the dataset first: click [titanic dataset.zip](https://github.com/devtlv/Datasets-DA-Bootcamp-2-/raw/refs/heads/main/Week%204%20-%20Data%20Understanding/W4D4%20-%20Data%20Preprocessing%20&%20T/titanic%20dataset.zip). It downloads `titanic dataset.zip`. Save it in `C:\Users\sivan\Datasets`.
3. When the upload cell in Exercise 1 asks for a file, choose `titanic dataset.zip` from `C:\Users\sivan\Datasets`. All 7 exercises use this one file, so you upload it only once.

**Run the cells from top to bottom** (Shift+Enter runs one cell). Every exercise keeps working on the same table `df`, so after a restart you must run again from the top.

# 👩‍🏫 👩🏿‍🏫 What You’ll learn

By the end of this notebook, you will be able to:

- Detect and remove duplicate entries from a dataset.
- Apply appropriate strategies for handling missing values.
- Perform basic feature engineering (new attributes, encoding).
- Detect and handle outliers using statistical methods.
- Standardize and normalize features to prepare for modeling.
- Transform and encode features (e.g. Age groups) for improved model input.

Each exercise builds on the previous one — keep a consistent preprocessing narrative throughout the notebook. Also, be sure to comment your code to explain your decisions, especially when choosing thresholds or methods.

💡 **Tip:** The recommended preprocessing sequence is:

1. Handle duplicates
2. Address missing values
3. Treat outliers
4. Encode categorical variables

For all of the below exercises, you will use the [Titanic dataset (train.csv)](https://github.com/devtlv/Datasets-DA-Bootcamp-2-/raw/refs/heads/main/Week%204%20-%20Data%20Understanding/W4D4%20-%20Data%20Preprocessing%20&%20T/titanic%20dataset.zip), so load it beforehand on your notebook.
You will notice in the following exercises that the dataset is already pretty clean but try and understand all of the functions used for preprocessing the data.
Optionally, if you have time and willing to, you can redo the exercises with a less clean dataset : [Weather Data Munich 1954-2022](https://github.com/devtlv/Datasets-DA-Bootcamp-2-/raw/refs/heads/main/Week%204%20-%20Data%20Understanding/W4D4%20-%20Data%20Preprocessing%20&%20T/Weather%20Data%20Munich%201954-2022.zip).

## Setup
Run this cell first. It imports the libraries used in all the exercises.

In [ ]:
import pandas as pd
# pandas: for working with tables (DataFrames)
import numpy as np
# numpy: for math on whole columns (we use it for the log transformation)
import matplotlib.pyplot as plt
# matplotlib: for drawing charts
import seaborn as sns
# seaborn: for nicer statistical charts (built on matplotlib)

---
# 🌟 Exercise 1: Duplicate Detection and Removal

**Instructions**

**Objective:** Identify and remove duplicate entries in the Titanic dataset.

- Load the Titanic dataset.
- Identify if there are any duplicate rows based on all columns.
- Remove any duplicate rows found in the dataset.
- Verify the removal of duplicates by checking the number of rows before and after the duplicate removal.

*Hint: Use the duplicated() and drop_duplicates() functions in Pandas.*

### Step 1: Load the Titanic dataset

In [ ]:
from google.colab import files
# the Colab tool for uploading files from your computer
uploaded = files.upload()
# choose "titanic dataset.zip" (saved in a variable so its contents are not printed)
!unzip "titanic dataset.zip"
# unzip it: you get the folder "titanic dataset" with train.csv, test.csv and gender_submission.csv

In [ ]:
df = pd.read_csv('titanic dataset/train.csv')
# load train.csv into a DataFrame called df (the exercise asks for train.csv)
print(df.shape)
# print (number of rows, number of columns)
print(df.head())
# show the first 5 rows

### Step 2: Identify duplicate rows (based on all columns)

In [ ]:
print('Duplicate rows (all columns):', df.duplicated().sum())
# duplicated() marks each row that is an exact copy of an earlier row with True; sum() counts the Trues
print('Duplicate rows (all columns except PassengerId):', df.drop(columns=['PassengerId']).duplicated().sum())
# PassengerId is different for every row, so it could hide copies; we check again without it

### Step 3: Remove duplicates and verify (rows before and after)

In [ ]:
rows_before = len(df)
# count the rows before removing duplicates
df = df.drop_duplicates()
# drop_duplicates() keeps the first copy of each row and removes the repeats
rows_after = len(df)
# count the rows after removing duplicates
print('Rows before:', rows_before)
# show the row count before
print('Rows after:', rows_after)
# show the row count after
print('Rows removed:', rows_before - rows_after)
# the difference is the number of duplicates removed

**Explanation:** The first print shows **0** duplicate rows, both with all columns and without `PassengerId`, so there were no copies to remove. The row counts confirm it: **891 rows before and 891 after** (0 removed). We still ran `drop_duplicates()` because it is the step the exercise asks for, and the before/after counts prove the table is clean.

---
# 🌟 Exercise 2: Handling Missing Values

**Instructions**

- Identify columns in the Titanic dataset with missing values.
- Explore different strategies for handling missing data, such as removal, imputation, and filling with a constant value.
- Apply each strategy to different columns based on the nature of the data.

*Hint: Review methods like dropna(), fillna(), and SimpleImputer from scikit-learn.*

### Step 1: Find the columns with missing values

In [ ]:
missing = df.isnull().sum()
# isnull() marks every empty cell with True; sum() counts them in each column
print(missing)
# show the count of missing values in every column

In [ ]:
missing_percent = (missing / len(df) * 100).round(1)
# turn the counts into a percentage of all rows, rounded to 1 decimal
print(missing_percent[missing_percent > 0])
# show only the columns that have missing values

### Step 2: Choose one strategy for each column
We look at each column with missing values and choose the strategy that fits it:
- **Cabin → fill with a constant value** (`fillna('Unknown')`): most of the column is empty, and an empty cabin is information in itself (the cabin was not recorded), so we mark it as "Unknown" instead of guessing a cabin.
- **Age → imputation with the median** (`SimpleImputer`): age is a number and only part of it is missing, so we fill the gaps with a typical age. We use the median and not the mean because the median is not pulled by very old or very young passengers.
- **Embarked → removal** (`dropna()`): only 2 rows are missing, so removing them loses almost nothing.

The prints below show the numbers behind each choice.

In [ ]:
print('Age median:', df['Age'].median())
# the middle age when all known ages are sorted
print('Age mean:', round(df['Age'].mean(), 2))
# the average age, for comparison with the median
print(df['Embarked'].value_counts())
# how many passengers boarded at each port (S, C, Q)

### Strategy 1: Fill Cabin with a constant value

In [ ]:
df['Cabin'] = df['Cabin'].fillna('Unknown')
# replace every empty Cabin with the text 'Unknown'
print('Missing Cabin after:', df['Cabin'].isnull().sum())
# check: no empty Cabin left
print('Cabins marked Unknown:', (df['Cabin'] == 'Unknown').sum())
# count how many rows got the constant value

### Strategy 2: Impute Age with the median (SimpleImputer)

In [ ]:
from sklearn.impute import SimpleImputer
# SimpleImputer is the scikit-learn tool that fills missing values
imputer = SimpleImputer(strategy='median')
# create an imputer that fills gaps with the median of the column
df[['Age']] = imputer.fit_transform(df[['Age']])
# fit learns the median of Age, transform fills the empty ages with it (double brackets = a table with one column)
print('Median used by the imputer:', imputer.statistics_[0])
# the value the imputer used to fill the gaps
print('Missing Age after:', df['Age'].isnull().sum())
# check: no empty Age left

### Strategy 3: Remove the rows with missing Embarked

In [ ]:
rows_before = len(df)
# count the rows before removing
df = df.dropna(subset=['Embarked'])
# remove only the rows where Embarked is empty
print('Rows before:', rows_before)
# rows before the removal
print('Rows after:', len(df))
# rows after the removal

### Check: no missing values left

In [ ]:
print(df.isnull().sum())
# every column should now show 0

**Explanation:** The `isnull().sum()` print shows three columns with missing values: **Age 177** (19.9%), **Cabin 687** (77.1%) and **Embarked 2** (0.2%). We used a different strategy for each one:
- **Cabin – constant value:** 77.1% is empty, too much to guess, so the 687 empty cells became `'Unknown'` (the print shows 687 cabins marked Unknown).
- **Age – imputation:** the print shows the median is **28.0** and the mean **29.7**; we used the median because extreme ages pull the mean, not the median. The imputer used 28.0 and 0 ages are missing after.
- **Embarked – removal:** only 2 rows were missing, so `dropna()` took the table from **891 to 889 rows**.

The last print shows **0** missing values in every column.

---
# 🌟 Exercise 3: Feature Engineering

**Instructions**

- Create new features, such as Family Size from SibSp and Parch, and Title extracted from the Name column.
- Convert categorical variables into numerical form using techniques like one-hot encoding or label encoding.
- You will encode new categorical features (like Title) here, but do not scale numerical features yet — that will come after outlier handling.

*Hint: Utilize Pandas for data manipulation and scikit-learn’s preprocessing module for encoding.*

### Step 1: Create FamilySize from SibSp and Parch

In [ ]:
df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
# SibSp = siblings/spouses on board, Parch = parents/children on board, +1 counts the passenger too
print(df[['SibSp', 'Parch', 'FamilySize']].head())
# show the first rows to check the sum
print(df['FamilySize'].value_counts().sort_index())
# how many passengers have each family size

### Step 2: Extract Title from the Name column

In [ ]:
print(df['Name'].head(3))
# every name looks like "Last name, Title. First names"

In [ ]:
after_comma = df['Name'].str.split(', ').str[1]
# split each name at ", " and keep the second part, e.g. "Mr. Owen Harris"
df['Title'] = after_comma.str.split('.').str[0]
# split that part at "." and keep the first piece, e.g. "Mr"
print(df['Title'].value_counts())
# count how many passengers have each title

Many titles appear only a few times. We group every title with fewer than 10 passengers into one group called `Rare`, so the encoding does not create a column for just 1 or 2 people.

In [ ]:
title_counts = df['Title'].value_counts()
# the number of passengers for each title
rare_titles = title_counts[title_counts < 10].index
# the titles that have fewer than 10 passengers
df['Title'] = df['Title'].replace(list(rare_titles), 'Rare')
# replace each of those rare titles with the word 'Rare'
print(df['Title'].value_counts())
# check the new groups

### Step 3: Encode Title – label encoding or one-hot encoding?
The exercise names two techniques. We try **label encoding** first on Title, to see what it does, and then choose.

In [ ]:
from sklearn.preprocessing import LabelEncoder
# LabelEncoder (from scikit-learn's preprocessing module) gives each category a number
title_encoder = LabelEncoder()
# create the encoder
title_labels = title_encoder.fit_transform(df['Title'])
# one number per passenger: the titles in alphabetical order get 0, 1, 2, ...
print(title_encoder.classes_.tolist())
# the titles in the order of their numbers (the first one is 0)
print(pd.Series(title_labels).value_counts().sort_index())
# how many passengers got each number

**Label or one-hot?** The print shows what label encoding did: **Master = 0, Miss = 1, Mr = 2, Mrs = 3, Rare = 4** (alphabetical order), with 40, 181, 517, 124 and 27 passengers. A model would read these numbers as an order and a size: "Rare (4) is more than Mr (2)" and "Mr is twice Miss". That is false, because titles have no order (Title is **nominal**). So we do **not** keep these numbers; we use **one-hot encoding** for Title (next cell). Label encoding fits only categories with a real order, and we use it in Exercise 6 for `Pclass`.

In [ ]:
title_dummies = pd.get_dummies(df['Title'], prefix='Title', dtype=int)
# make one 0/1 column per title (Title_Master, Title_Miss, ...); dtype=int gives 0 and 1 instead of False and True
df = pd.concat([df, title_dummies], axis=1)
# add the new columns to the right of df (axis=1 means side by side)
print(df[['Name', 'Title', 'Title_Master', 'Title_Miss', 'Title_Mr', 'Title_Mrs', 'Title_Rare']].head())
# check: each row has a 1 only in its own title column
print(df.shape)
# the new size of the table

**Explanation:** `FamilySize = SibSp + Parch + 1`: the print shows that most passengers travelled alone (**535** with FamilySize 1) and the largest family has **11** people. The Title came out of the name in two splits; the first count shows 17 different titles, but many have only 1 or 2 passengers, so the ones with fewer than 10 were grouped into **Rare (27 passengers)**, next to Mr 517, Miss 181, Mrs 124 and Master 40. Title has no order, so (as the label-encoding test showed) we used one-hot encoding (`get_dummies`): 5 new 0/1 columns, and the table now has **889 rows and 19 columns**. No number was scaled here, as the exercise asks.

---
# 🌟 Exercise 4: Outlier Detection and Handling

**Goal:** Detect and cap or transform outliers in columns like Fare and Age.

1. Visualize distributions using boxplots or histograms to identify potential outliers.
2. Use IQR or Z-score methods to detect them.
3. Handle outliers with:
   - Quantile capping (e.g. 0.98)
   - Log transformation
   - Row removal
4. Compare the dataset before and after treatment.

📌 **Note:** Small differences between 0.98 and 0.99 quantiles are normal when extreme values are rare or far apart. Use df.quantile() to explore and choose thresholds empirically, backed by visualization.

### Step 1: Visualize Fare and Age (boxplots and histograms)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
# make a grid of 2 x 2 charts
sns.boxplot(x=df['Fare'], ax=axes[0, 0])
# boxplot of Fare: the dots on the right are possible outliers
axes[0, 0].set_title('Fare - boxplot')
# title of the first chart
sns.histplot(df['Fare'], bins=40, ax=axes[0, 1])
# histogram of Fare: how many passengers paid each price
axes[0, 1].set_title('Fare - histogram')
# title of the second chart
sns.boxplot(x=df['Age'], ax=axes[1, 0])
# boxplot of Age
axes[1, 0].set_title('Age - boxplot')
# title of the third chart
sns.histplot(df['Age'], bins=40, ax=axes[1, 1])
# histogram of Age
axes[1, 1].set_title('Age - histogram')
# title of the fourth chart
plt.tight_layout()
# leave space so the titles do not overlap
plt.show()
# show the charts

In [ ]:
print(df[['Fare', 'Age']].describe())
# count, mean, std, min, quartiles and max of Fare and Age

### Step 2a: Detect outliers with the IQR method

In [ ]:
for col in ['Fare', 'Age']:
# repeat the same steps for Fare and then for Age
    Q1 = df[col].quantile(0.25)
    # Q1: the value a quarter of the way up the sorted column
    Q3 = df[col].quantile(0.75)
    # Q3: the value three quarters of the way up
    IQR = Q3 - Q1
    # IQR: the distance between Q1 and Q3 (the middle half of the data)
    lower = Q1 - 1.5 * IQR
    # values below this are outliers
    upper = Q3 + 1.5 * IQR
    # values above this are outliers
    outliers = df[(df[col] < lower) | (df[col] > upper)]
    # keep the rows below the lower limit OR (|) above the upper limit
    print(col, '| lower:', round(lower, 2), '| upper:', round(upper, 2), '| IQR outliers:', len(outliers))
    # show the limits and how many outliers they find

### Step 2b: Detect outliers with the Z-score method

In [ ]:
from scipy import stats
# scipy.stats has the zscore function
for col in ['Fare', 'Age']:
# repeat for Fare and Age
    z = np.abs(stats.zscore(df[col]))
    # z-score = how many standard deviations a value is from the mean; abs() removes the minus sign
    # Another way, by hand (same result): z = (value - mean) / std
    # z = np.abs((df[col] - df[col].mean()) / df[col].std(ddof=0))
    print(col, '| Z-score outliers (|z| > 3):', (z > 3).sum())
    # count the values more than 3 standard deviations away from the mean

### Step 2c: Look at the high quantiles to choose the capping threshold

In [ ]:
print(df['Fare'].quantile([0.95, 0.98, 0.99, 1.0]))
# the Fare value below which 95%, 98%, 99% and 100% of passengers are
print('Passengers above the 0.98 quantile:', (df['Fare'] > df['Fare'].quantile(0.98)).sum())
# how many fares the 0.98 cap would change
print('Passengers above the 0.99 quantile:', (df['Fare'] > df['Fare'].quantile(0.99)).sum())
# how many fares the 0.99 cap would change

**What the detection shows:**
- The boxplot and histogram of **Fare** have a long tail to the right: the `describe()` print shows a median of 14.45 but a max of **512.33**.
- **IQR** finds **114** Fare outliers (above 65.66) and **65** Age outliers (below 2.5 or above 54.5). The Age count is high because the 177 filled ages all became 28, which makes the middle of Age very narrow, so normal ages like 55 or 60 fall outside it.
- **Z-score** (|z| > 3) is stricter: **20** Fare outliers and **7** Age outliers.
- **Quantiles:** the 0.98 quantile is **211.34** and the 0.99 quantile is **249.30**, while the max is 512.33. Capping at 0.98 changes **17** fares and at 0.99 only **9**. We choose **0.98**, as the exercise suggests: the 0.99 cap still keeps fares far above the rest of the data, and 17 out of 889 is still a small change.

**Decision for Age:** the ages are real and possible (the oldest is 80, the youngest 0.42 = a baby), and the Z-score finds only 7, so we do **not** treat Age. We treat **Fare** only.

### Step 3: Handle the Fare outliers with three methods
Each method works on its own copy of df, so we can compare all three with the original before we choose one.

**Method 1: Quantile capping at 0.98**

In [ ]:
df_capped = df.copy()
# a copy of df for this method
fare_cap = df['Fare'].quantile(0.98)
# the 0.98 quantile of Fare (98% of fares are below it)
df_capped['Fare'] = df_capped['Fare'].clip(upper=fare_cap)
# clip(upper=...) replaces every fare above the cap with the cap value
print('Cap value:', round(fare_cap, 2))
# the value used as the cap
print('Max Fare before:', df['Fare'].max(), '| after:', df_capped['Fare'].max())
# the highest fare before and after capping

**Method 2: Log transformation**

In [ ]:
df_log = df.copy()
# a copy of df for this method
df_log['Fare'] = np.log1p(df_log['Fare'])
# log1p(x) = log(1 + x): it squeezes big values much more than small ones; the +1 avoids log(0) for free tickets
print('Fare before (first 5):', df['Fare'].head().round(2).tolist())
# the first 5 fares before
print('Fare after log (first 5):', df_log['Fare'].head().round(2).tolist())
# the same 5 fares after the log

**Method 3: Row removal (IQR limits)**

In [ ]:
Q1 = df['Fare'].quantile(0.25)
# Q1 of Fare
Q3 = df['Fare'].quantile(0.75)
# Q3 of Fare
IQR = Q3 - Q1
# IQR of Fare
upper = Q3 + 1.5 * IQR
# the upper IQR limit
df_removed = df[df['Fare'] <= upper]
# keep only the rows with a fare up to the upper limit (the lower limit is below 0, so no fare is under it)
print('Rows before:', len(df), '| after removal:', len(df_removed), '| removed:', len(df) - len(df_removed))
# how many rows the removal deleted

### Step 4: Compare the dataset before and after each treatment

In [ ]:
comparison = pd.DataFrame({
    'original': df['Fare'].describe(),
    'capped_0.98': df_capped['Fare'].describe(),
    'log': df_log['Fare'].describe(),
    'rows_removed': df_removed['Fare'].describe()
})
# put the describe() of Fare from each version side by side in one table
print(comparison.round(2))
# show the table

In [ ]:
print('Skew original:', round(df['Fare'].skew(), 2))
# skew = how lopsided the distribution is (0 = symmetric, big positive = long tail of high values)
print('Skew capped 0.98:', round(df_capped['Fare'].skew(), 2))
# skew after capping
print('Skew log:', round(df_log['Fare'].skew(), 2))
# skew after the log
print('Skew rows removed:', round(df_removed['Fare'].skew(), 2))
# skew after removing rows

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 3))
# one row of 4 charts
sns.boxplot(x=df['Fare'], ax=axes[0])
# original Fare
axes[0].set_title('Original')
# chart title
sns.boxplot(x=df_capped['Fare'], ax=axes[1])
# Fare after capping
axes[1].set_title('Capped at 0.98')
# chart title
sns.boxplot(x=df_log['Fare'], ax=axes[2])
# Fare after the log
axes[2].set_title('Log transformed')
# chart title
sns.boxplot(x=df_removed['Fare'], ax=axes[3])
# Fare after removing rows
axes[3].set_title('Rows removed (IQR)')
# chart title
plt.tight_layout()
# leave space between the charts
plt.show()
# show the charts

**Comparison and choice:** the table and the skew prints show what each method did to Fare:
- **Capping at 0.98:** all **889** rows are kept, the max falls from 512.33 to **211.34**, and the skew falls from **4.8 to 2.81**. The fares stay in pounds.
- **Log:** the skew falls the most, to **0.4**, but the values are no longer prices (the max is 6.24), which makes them harder to read.
- **Row removal:** the skew falls to **1.43**, but **114 rows** (12.8% of the passengers) are deleted, most of them real 1st-class passengers.

**We carry the capped version (0.98) forward**: it keeps every passenger and keeps Fare in real prices, and it still cuts the extreme values. The next cell makes it the new `df`.

In [ ]:
df = df_capped.copy()
# carry the capped version forward: from now on df has Fare capped at the 0.98 quantile
print(df.shape)
# all rows are still here
print('Max Fare now:', df['Fare'].max())
# the highest fare is now the cap value

---
# 🌟 Exercise 5: Data Standardization and Normalization

**Goal:** Scale numerical features to prepare for modeling.

1. Use StandardScaler (mean = 0, std = 1) for normally distributed features.
2. Use MinMaxScaler (range [0, 1]) for features that are skewed or bounded.

📌 **Important:** Perform this step after outlier treatment to avoid distortion caused by extreme values.

### Step 1: Choose which scaler fits each number column
The number columns we scale are `Age`, `Fare` and `FamilySize`. We print the skew of each one to decide:
- a column that is close to symmetric (skew near 0) gets **StandardScaler** (mean 0, std 1), which works best for bell-shaped data;
- a column that is still lopsided gets **MinMaxScaler** (range 0 to 1), which keeps every value between fixed limits and does not assume a bell shape.

In [ ]:
print(df[['Age', 'Fare', 'FamilySize']].skew().round(2))
# skew of each number column (near 0 = symmetric, above 1 = strongly lopsided)

**Choice:** the skew print shows **Age 0.51** (close to symmetric), so Age gets **StandardScaler**. **Fare 2.81** and **FamilySize 2.72** are still strongly lopsided, so they get **MinMaxScaler** into the range 0 to 1.

### Step 2: StandardScaler on Age

In [ ]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler
# the two scalers from scikit-learn
standard_scaler = StandardScaler()
# create the standard scaler
df['Age_scaled'] = standard_scaler.fit_transform(df[['Age']])
# fit learns the mean and std of Age, transform turns each age into (age - mean) / std; saved in a new column
print('Age_scaled mean:', round(df['Age_scaled'].mean(), 2))
# should be 0
print('Age_scaled std:', round(df['Age_scaled'].std(ddof=0), 2))
# should be 1 (ddof=0 uses the same formula as the scaler)

### Step 3: MinMaxScaler on Fare and FamilySize

In [ ]:
minmax_scaler = MinMaxScaler()
# create the min-max scaler
df[['Fare_scaled', 'FamilySize_scaled']] = minmax_scaler.fit_transform(df[['Fare', 'FamilySize']])
# fit learns the min and max of each column, transform turns each value into (value - min) / (max - min)
print(df[['Fare_scaled', 'FamilySize_scaled']].describe().round(2))
# min should be 0 and max should be 1 for both

We keep the original `Age`, `Fare` and `FamilySize` columns next to the scaled ones, because Exercise 7 needs the real ages (in years) to make the age groups.

In [ ]:
print(df[['Age', 'Age_scaled', 'Fare', 'Fare_scaled', 'FamilySize', 'FamilySize_scaled']].head())
# compare the original and scaled values side by side

**Explanation:** After `StandardScaler`, the print shows Age_scaled has **mean 0.0 and std 1.0**: each value now says how many standard deviations a passenger is from the average age (for example 22 years → -0.56). After `MinMaxScaler`, the `describe()` print shows **min 0.00 and max 1.00** for both Fare_scaled and FamilySize_scaled. The scaled values are in new columns, so the original Age, Fare and FamilySize are still there (the last print shows them side by side).

---
# 🌟 Exercise 6: Feature Encoding

**Goal:** Finalize categorical variable encoding.

1. Identify remaining categorical columns (e.g. Sex, Embarked, Title).
2. Apply:
   - One-Hot Encoding for nominal variables.
   - Label Encoding if any ordinal variables remain.
3. Merge encoded columns back into the main dataset.

📌 **Reminder:** Encoding comes after handling missing values and outliers, but before scaling (if applicable).

### Step 1: Find the categorical (text) columns

In [ ]:
print(df.select_dtypes(include='object').columns.tolist())
# list the columns whose type is text (object)
print(df[['Sex', 'Embarked', 'Title', 'Pclass']].nunique())
# how many different values Sex, Embarked, Title and Pclass have
print(df.filter(like='Title_').columns.tolist())
# the one-hot Title columns we already made in Exercise 3

**Choice:** the first print shows the text columns: Name, Sex, Ticket, Cabin, Embarked and Title. **Title** has 5 values and is **already encoded**: the last print lists the 5 one-hot columns made in Exercise 3 (Title_Master ... Title_Rare). Name, Ticket and Cabin are free text, not categories. What remains:
- **Sex** (**2** values) and **Embarked** (**3** ports): no order (S is not "more" than C), so they are **nominal** → **one-hot encoding**.
- **Pclass** (**3** classes): a real order (1st > 2nd > 3rd), so it is **ordinal** → **label encoding**. Pclass is stored as a number, not text, which is why it is not in the first list.

### Step 2: One-hot encoding for the nominal columns Sex and Embarked

In [ ]:
nominal_dummies = pd.get_dummies(df[['Sex', 'Embarked']], prefix=['Sex', 'Embarked'], dtype=int)
# one 0/1 column per value: Sex_female, Sex_male, Embarked_C, Embarked_Q, Embarked_S
print(nominal_dummies.head())
# check: in each row there is one 1 for Sex and one 1 for Embarked

### Step 3: Label encoding for the ordinal column Pclass

In [ ]:
from sklearn.preprocessing import LabelEncoder
# LabelEncoder turns each category into a number
pclass_encoder = LabelEncoder()
# create the encoder
pclass_encoded = pclass_encoder.fit_transform(df['Pclass'])
# 1st class -> 0, 2nd class -> 1, 3rd class -> 2 (the order of the classes is kept)
print(pclass_encoder.classes_.tolist())
# the classes in the order of their numbers (the first one is 0)
print(pd.Series(pclass_encoded).value_counts().sort_index())
# how many passengers got each number

### Step 4: Merge the encoded columns back into df

In [ ]:
columns_before = df.shape[1]
# the number of columns before the merge
df = pd.concat([df, nominal_dummies], axis=1)
# add the 5 one-hot columns to the right of df (axis=1 = side by side, rows are matched by their index)
df['Pclass_encoded'] = pclass_encoded
# add the label-encoded Pclass as a new column
print('Columns before:', columns_before, '| after:', df.shape[1])
# 5 one-hot columns + 1 label column = 6 new columns
print(df[['Sex', 'Sex_female', 'Sex_male', 'Embarked', 'Embarked_C', 'Embarked_Q', 'Embarked_S', 'Pclass', 'Pclass_encoded']].head())
# the original columns next to their encoded versions

**Explanation:**
- **One-hot:** `get_dummies` made **Sex_female, Sex_male, Embarked_C, Embarked_Q, Embarked_S**. The print shows exactly one 1 for Sex and one 1 for Embarked in each row (row 0 is a male from S: Sex_male 1, Embarked_S 1).
- **Label:** `LabelEncoder` printed its classes `[1, 2, 3]`, so **1st class = 0, 2nd = 1, 3rd = 2**. The order is kept, and the counts (214, 184, 491) are the same as before.
- **Merge back:** `pd.concat(..., axis=1)` and the new `Pclass_encoded` column added them to `df`: **22 columns before, 28 after** (5 one-hot + 1 label). The last print shows each original column next to its encoded version.
- **About the reminder (encoding before scaling):** we scaled in Exercise 5, before this exercise, because we follow the exercise order. It makes no difference here: the scalers only touched Age, Fare and FamilySize, and the encoding only touches Sex, Embarked and Pclass, and the 0/1 columns do not need scaling.

---
# 🌟 Exercise 7: Data Transformation for Age Feature

**Goal:** Create and encode age groups.

1. Use pd.cut() to create bins for life stages (e.g. child, teen, adult, senior).
2. Apply one-hot encoding using pd.get_dummies().

📌 **Example:** You might define bins like [0, 12, 18, 60, 100] and label them accordingly.

### Step 1: Create age groups with pd.cut
We use the bins from the exercise, **[0, 12, 18, 60, 100]**, for 4 life stages: Child (up to 12), Teen (over 12 up to 18), Adult (over 18 up to 60) and Senior (over 60). The print below shows the youngest and oldest ages, to check that every age fits inside 0 to 100.

In [ ]:
print('Youngest:', df['Age'].min(), '| Oldest:', df['Age'].max())
# the smallest and largest age, to check that the bins cover everyone

In [ ]:
bins = [0, 12, 18, 60, 100]
# the edges of the groups: (0-12], (12-18], (18-60], (60-100]
labels = ['Child', 'Teen', 'Adult', 'Senior']
# one name for each group (4 names for 5 edges)
df['AgeGroup'] = pd.cut(df['Age'], bins=bins, labels=labels)
# pd.cut puts each age into the group whose edges it falls between
print(df['AgeGroup'].value_counts().sort_index())
# how many passengers are in each group, in age order

In [ ]:
print(df.groupby('AgeGroup', observed=True)['Survived'].mean().round(2))
# the survival rate of each age group (Survived is 0 or 1, so the mean is the share who survived)

### Step 2: One-hot encode the age groups

In [ ]:
age_dummies = pd.get_dummies(df['AgeGroup'], prefix='AgeGroup', dtype=int)
# one 0/1 column per age group
df = pd.concat([df, age_dummies], axis=1)
# add them to df
print(df[['Age', 'AgeGroup', 'AgeGroup_Child', 'AgeGroup_Teen', 'AgeGroup_Adult', 'AgeGroup_Senior']].head())
# check: each row has a 1 only in its own age group

### Step 3: The final table, ready for a model

A model needs only numbers, so we drop the text columns we already encoded (or that cannot be used as they are) and keep the rest.

In [ ]:
model_df = df.drop(columns=['PassengerId', 'Name', 'Ticket', 'Cabin', 'Sex', 'Embarked', 'Title', 'AgeGroup', 'Pclass'])
# remove the ID, the free-text columns and the original category columns (their encoded versions stay)
print(model_df.shape)
# the size of the final table
print(model_df.dtypes)
# every column should now be a number (int or float)

**Explanation:** The ages go from **0.42 to 80.0**, so the bins **[0, 12, 18, 60, 100]** from the exercise cover everyone. `pd.cut` made 4 groups: **Adult 729**, Teen 70, Child 69 and Senior 21. Adult is by far the biggest group: it is a wide bin (19 to 60), and the 177 filled ages (all 28) are in it too. The survival print shows why the groups are useful: **58%** of children survived, **43%** of teens, **36%** of adults and only **19%** of seniors. `get_dummies` turned the groups into 4 binary 0/1 columns. The last print shows the final table: **889 rows and 24 columns, all numbers** (int64 or float64), so it is ready for a model.

---
**Optional part:** redoing the exercises with the [Weather Data Munich 1954-2022](https://github.com/devtlv/Datasets-DA-Bootcamp-2-/raw/refs/heads/main/Week%204%20-%20Data%20Understanding/W4D4%20-%20Data%20Preprocessing%20&%20T/Weather%20Data%20Munich%201954-2022.zip) dataset is optional, so it was skipped here.